# Model complexity and configurable-batch inference speed

This notebook measures **parameter count**, **model size**, **GFLOPs per image**, **FPS**, batch latency, per-image latency, and peak inference memory using a configurable inference batch size. It does not load a dataset, train a model, or change saved predictions. GFLOPs are always measured with one synthetic image; speed is measured with the batch size selected below.

Results are written to `model_complexity_batch_outputs/<timestamp>_batch_<size>_complexity/`. Nothing is written to or changed inside `benchmark_outputs/`.

The default PyTorch profiler counts FLOPs for operations it supports (principally convolutions and matrix multiplications). The output records the backend and convention. For a paper, use the same backend for every model and state it in the table caption or methods section.

In [ ]:
# Run settings
EXPERIMENT_FOLDER = 'all'  # 'latest', 'all', an exact folder name, or a short match
INFERENCE_BATCH_SIZE = 4
PROFILE_DEVICE_NAME = 'cuda:0'  # 'auto', 'cpu', or 'cuda:0'
FLOP_BACKEND = 'torch_profiler'  # 'torch_profiler' or 'fvcore'
REPRESENTATIVE_SEED = 42
FPS_WARMUP_ITERATIONS = 30
FPS_TIMED_ITERATIONS = 100
FPS_REPEATS = 5

In [ ]:
from collections import Counter
from datetime import datetime
from functools import partial
from pathlib import Path
import json
import os
import platform
import statistics
import sys
import time
import traceback

import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.profiler import ProfilerActivity, profile

start = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (p for p in (start, *start.parents)
     if (p / 'benchmark_outputs').is_dir()
     and (p / 'SARATR-X').is_dir()
     and (p / 'SAR-JEPA').is_dir()),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the COMPASS_NET_PAPER workspace.')

BENCHMARK_ROOT = WORKSPACE_ROOT / 'benchmark_outputs'
output_override = os.environ.get('MODEL_BATCH_PROFILE_OUTPUT_ROOT', '').strip()
OUTPUT_ROOT = (
    Path(output_override).expanduser().resolve()
    if output_override else WORKSPACE_ROOT / 'model_complexity_batch_outputs'
)
device_name = os.environ.get('MODEL_PROFILE_DEVICE', PROFILE_DEVICE_NAME).strip()
if device_name == 'auto':
    device_name = 'cuda:0' if torch.cuda.is_available() else 'cpu'
PROFILE_DEVICE = torch.device(device_name)
if PROFILE_DEVICE.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA was selected but is not available in this kernel.')

# Environment overrides are used by the automated notebook check.
limit_text = os.environ.get('MODEL_PROFILE_LIMIT', '').strip()
MAX_EXPERIMENTS = int(limit_text) if limit_text else None
experiment_override = os.environ.get('MODEL_PROFILE_MATCH', '').strip()
if experiment_override:
    EXPERIMENT_FOLDER = experiment_override
INFERENCE_BATCH_SIZE = int(
    os.environ.get('MODEL_PROFILE_BATCH_SIZE', INFERENCE_BATCH_SIZE)
)
if INFERENCE_BATCH_SIZE < 1:
    raise ValueError('INFERENCE_BATCH_SIZE must be at least 1.')
FPS_WARMUP_ITERATIONS = int(
    os.environ.get('MODEL_PROFILE_FPS_WARMUP', FPS_WARMUP_ITERATIONS)
)
FPS_TIMED_ITERATIONS = int(
    os.environ.get('MODEL_PROFILE_FPS_ITERATIONS', FPS_TIMED_ITERATIONS)
)
FPS_REPEATS = int(os.environ.get('MODEL_PROFILE_FPS_REPEATS', FPS_REPEATS))

try:
    import fvcore
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except ImportError:
    fvcore = None
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False

print(f'Workspace: {WORKSPACE_ROOT}')
print(f'Benchmark input: {BENCHMARK_ROOT}')
print(f'Complexity output: {OUTPUT_ROOT}')
print(f'Experiment selection: {EXPERIMENT_FOLDER}')
print(f'Inference batch size: {INFERENCE_BATCH_SIZE}')
print(f'Device: {PROFILE_DEVICE}')
print(f'FLOP backend: {FLOP_BACKEND}')
print(f'fvcore available: {FVCORE_AVAILABLE}')

## Optional fvcore backend

The notebook works without additional packages using `torch_profiler`. If you prefer fvcore, install it once in this kernel, restart the kernel, change `FLOP_BACKEND` above to `"fvcore"`, and run all cells:

```python
%pip install fvcore
```

Do not combine values from different profiler backends in one comparison table.

In [ ]:
def model_input_channels(config):
    input_channels = int(config['input_channels'])
    if input_channels == 2:
        return 2
    return 3 if config['model'] == 'saratrx' else 1


def build_model(config):
    model_name = config['model']
    image_size = int(config['image_size'])
    in_channels = model_input_channels(config)
    num_classes = len(config['classes'])

    if model_name == 'saratrx':
        source_path = WORKSPACE_ROOT / 'SARATR-X' / 'classification'
        if str(source_path) not in sys.path:
            sys.path.insert(0, str(source_path))
        from model.hivit import HiViT
        return HiViT(
            img_size=image_size, patch_size=16, embed_dim=512, depths=[2, 2, 20],
            num_heads=8, stem_mlp_ratio=3.0, in_chans=in_channels, mlp_ratio=4.0,
            num_classes=num_classes, ape=True, rpe=False,
            norm_layer=partial(nn.LayerNorm, eps=1e-6),
        )

    if model_name == 'sarjepa':
        source_path = WORKSPACE_ROOT / 'SAR-JEPA' / 'few_shot_classification' / 'finetune'
        if str(source_path) not in sys.path:
            sys.path.insert(0, str(source_path))
        from vision_transformer_irpe import VisionTransformer
        return VisionTransformer(
            img_size=image_size, patch_size=16, embed_dim=768, in_chans=in_channels,
            num_classes=num_classes, depth=12, num_heads=12, mlp_ratio=4,
            qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6),
        )

    raise ValueError(f'Unsupported model: {model_name}')


def configure_trainable_parameters(model, config):
    model_name = config['model']
    training_mode = config['training_mode']

    if training_mode in {'linear_probe', 'partial_finetune'}:
        head = model.head
        model.head = nn.Sequential(
            nn.BatchNorm1d(head.in_features, affine=False, eps=1e-6), head,
        )

    if training_mode == 'full_finetune':
        for parameter in model.parameters():
            parameter.requires_grad = True
        return model

    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True

    final_norm = model.fc_norm if model_name == 'saratrx' else model.norm
    if model_name == 'saratrx':
        for parameter in final_norm.parameters():
            parameter.requires_grad = True

    if training_mode == 'partial_finetune':
        for block in model.blocks[-4:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in final_norm.parameters():
            parameter.requires_grad = True
    return model


def clear_rpe_cache(model):
    for module in model.modules():
        if hasattr(module, '_rp_bucket_buf'):
            module._rp_bucket_buf = (None, None, None)


def load_saved_model(config, checkpoint_path):
    model = configure_trainable_parameters(build_model(config), config)
    payload = torch.load(checkpoint_path, map_location='cpu')
    state_dict = payload.get('model_state_dict', payload)
    model.load_state_dict(state_dict, strict=True)
    return model, payload, state_dict

In [ ]:
def tensor_storage_bytes(tensors):
    return sum(
        value.numel() * value.element_size()
        for value in tensors
        if torch.is_tensor(value)
    )


def profile_with_torch(model, dummy_input):
    activities = [ProfilerActivity.CPU]
    if dummy_input.device.type == 'cuda':
        activities.append(ProfilerActivity.CUDA)

    with torch.inference_mode():
        model(dummy_input)  # warm-up and initialize model caches
        if dummy_input.device.type == 'cuda':
            torch.cuda.synchronize(dummy_input.device)
        with profile(activities=activities, with_flops=True, record_shapes=True) as prof:
            model(dummy_input)
        if dummy_input.device.type == 'cuda':
            torch.cuda.synchronize(dummy_input.device)

    events = prof.key_averages()
    counted = {event.key: int(event.flops) for event in events if event.flops}
    total_flops = sum(counted.values())
    if total_flops <= 0:
        raise RuntimeError('PyTorch profiler did not count any FLOPs for this model.')
    return {
        'flops': total_flops,
        'backend': 'torch_profiler',
        'convention': 'PyTorch supported-operator FLOPs; multiply and add are counted separately',
        'unsupported_ops': {},
        'counted_ops': counted,
    }


def profile_with_fvcore(model, dummy_input):
    if not FVCORE_AVAILABLE:
        raise ImportError('fvcore is not installed. Run `%pip install fvcore`, restart, and rerun.')
    analysis = FlopCountAnalysis(model, dummy_input)
    analysis.unsupported_ops_warnings(False)
    analysis.uncalled_modules_warnings(False)
    analysis.tracer_warnings('none')
    total_flops = int(analysis.total())
    return {
        'flops': total_flops,
        'backend': 'fvcore',
        'convention': 'fvcore convention: one fused multiply-add is counted as one FLOP',
        'unsupported_ops': {str(k): int(v) for k, v in analysis.unsupported_ops().items()},
        'counted_ops': {},
    }


def profile_flops(model, dummy_input):
    if FLOP_BACKEND == 'torch_profiler':
        return profile_with_torch(model, dummy_input)
    if FLOP_BACKEND == 'fvcore':
        return profile_with_fvcore(model, dummy_input)
    raise ValueError(f'Unknown FLOP_BACKEND: {FLOP_BACKEND}')


def benchmark_inference(model, dummy_input):
    batch_size = int(dummy_input.shape[0])
    def synchronize():
        if dummy_input.device.type == 'cuda':
            torch.cuda.synchronize(dummy_input.device)

    with torch.inference_mode():
        for _ in range(FPS_WARMUP_ITERATIONS):
            model(dummy_input)
        synchronize()

        if dummy_input.device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(dummy_input.device)

        batch_latencies_ms = []
        per_image_latencies_ms = []
        fps_values = []
        for _ in range(FPS_REPEATS):
            synchronize()
            started = time.perf_counter()
            for _ in range(FPS_TIMED_ITERATIONS):
                model(dummy_input)
            synchronize()
            elapsed = time.perf_counter() - started
            batch_latency_ms = 1000.0 * elapsed / FPS_TIMED_ITERATIONS
            batch_latencies_ms.append(batch_latency_ms)
            per_image_latencies_ms.append(batch_latency_ms / batch_size)
            fps_values.append(1000.0 * batch_size / batch_latency_ms)

    peak_memory = (
        torch.cuda.max_memory_allocated(dummy_input.device) / 1024**2
        if dummy_input.device.type == 'cuda' else None
    )
    return {
        'fps_mean': statistics.mean(fps_values),
        'fps_std': statistics.stdev(fps_values) if len(fps_values) > 1 else 0.0,
        'batch_latency_ms_mean': statistics.mean(batch_latencies_ms),
        'batch_latency_ms_std': (
            statistics.stdev(batch_latencies_ms) if len(batch_latencies_ms) > 1 else 0.0
        ),
        'per_image_latency_ms_mean': statistics.mean(per_image_latencies_ms),
        'per_image_latency_ms_std': (
            statistics.stdev(per_image_latencies_ms)
            if len(per_image_latencies_ms) > 1 else 0.0
        ),
        'fps_values': fps_values,
        'batch_latency_ms_values': batch_latencies_ms,
        'per_image_latency_ms_values': per_image_latencies_ms,
        'peak_inference_memory_MiB': peak_memory,
    }


def find_representative_checkpoint(experiment_dir):
    preferred = experiment_dir / 'runs' / f'seed_{REPRESENTATIVE_SEED}' / 'best_model.pth'
    if preferred.is_file():
        return preferred
    candidates = sorted((experiment_dir / 'runs').glob('seed_*/best_model.pth'))
    if not candidates:
        raise FileNotFoundError(f'No best_model.pth under {experiment_dir}')
    return candidates[0]


def profile_experiment(experiment_dir):
    config_path = experiment_dir / 'config.json'
    config = json.loads(config_path.read_text(encoding='utf-8'))
    checkpoint_path = find_representative_checkpoint(experiment_dir)
    model, payload, state_dict = load_saved_model(config, checkpoint_path)
    clear_rpe_cache(model)

    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    parameter_bytes = tensor_storage_bytes(model.parameters())
    buffer_bytes = tensor_storage_bytes(model.buffers())
    state_dict_bytes = tensor_storage_bytes(state_dict.values())

    in_channels = model_input_channels(config)
    image_size = int(config['image_size'])
    model = model.eval().to(PROFILE_DEVICE)
    flop_input = torch.zeros(
        1, in_channels, image_size, image_size,
        dtype=torch.float32, device=PROFILE_DEVICE,
    )
    speed_input = torch.zeros(
        INFERENCE_BATCH_SIZE, in_channels, image_size, image_size,
        dtype=torch.float32, device=PROFILE_DEVICE,
    )
    flop_report = profile_flops(model, flop_input)
    speed_report = benchmark_inference(model, speed_input)

    seed_name = checkpoint_path.parent.name
    seed = int(seed_name.removeprefix('seed_')) if seed_name.startswith('seed_') else None
    result = {
        'experiment': experiment_dir.name,
        'model': config['model'],
        'goal': config['goal'],
        'training_mode': config['training_mode'],
        'representative_seed': seed,
        'num_classes': len(config['classes']),
        'classes': json.dumps(config['classes']),
        'batch_size_profiled': INFERENCE_BATCH_SIZE,
        'input_channels_dataset': int(config['input_channels']),
        'input_channels_model': in_channels,
        'image_height': image_size,
        'image_width': image_size,
        'input_shape': (
            f'{INFERENCE_BATCH_SIZE}x{in_channels}x{image_size}x{image_size}'
        ),
        'flop_input_shape': f'1x{in_channels}x{image_size}x{image_size}',
        'total_parameters': total_parameters,
        'total_parameters_M': total_parameters / 1e6,
        'trainable_parameters': trainable_parameters,
        'trainable_parameters_M': trainable_parameters / 1e6,
        'trainable_percent': 100.0 * trainable_parameters / total_parameters,
        'frozen_parameters': total_parameters - trainable_parameters,
        'parameter_storage_MiB': parameter_bytes / 1024**2,
        'buffer_storage_MiB': buffer_bytes / 1024**2,
        'state_dict_tensor_storage_MiB': state_dict_bytes / 1024**2,
        'checkpoint_file_MiB': checkpoint_path.stat().st_size / 1024**2,
        'flops_per_image': flop_report['flops'],
        'GFLOPs_per_image': flop_report['flops'] / 1e9,
        'flop_backend': flop_report['backend'],
        'flop_convention': flop_report['convention'],
        'unsupported_ops': json.dumps(flop_report['unsupported_ops'], sort_keys=True),
        'counted_ops': json.dumps(flop_report['counted_ops'], sort_keys=True),
        'inference_device': str(PROFILE_DEVICE),
        'inference_dtype': str(speed_input.dtype).removeprefix('torch.'),
        'fps_warmup_iterations': FPS_WARMUP_ITERATIONS,
        'fps_timed_iterations': FPS_TIMED_ITERATIONS,
        'fps_repeats': FPS_REPEATS,
        'FPS_mean': speed_report['fps_mean'],
        'FPS_std': speed_report['fps_std'],
        'FPS_mean_std': f"{speed_report['fps_mean']:.2f} ± {speed_report['fps_std']:.2f}",
        'batch_latency_ms_mean': speed_report['batch_latency_ms_mean'],
        'batch_latency_ms_std': speed_report['batch_latency_ms_std'],
        'batch_latency_ms_mean_std': (
            f"{speed_report['batch_latency_ms_mean']:.3f} ± "
            f"{speed_report['batch_latency_ms_std']:.3f}"
        ),
        'per_image_latency_ms_mean': speed_report['per_image_latency_ms_mean'],
        'per_image_latency_ms_std': speed_report['per_image_latency_ms_std'],
        'per_image_latency_ms_mean_std': (
            f"{speed_report['per_image_latency_ms_mean']:.3f} ± "
            f"{speed_report['per_image_latency_ms_std']:.3f}"
        ),
        'peak_inference_memory_MiB': speed_report['peak_inference_memory_MiB'],
        'fps_values': json.dumps(speed_report['fps_values']),
        'batch_latency_ms_values': json.dumps(speed_report['batch_latency_ms_values']),
        'per_image_latency_ms_values': json.dumps(
            speed_report['per_image_latency_ms_values']
        ),
        'checkpoint': str(checkpoint_path.relative_to(WORKSPACE_ROOT)),
        'checkpoint_model': payload.get('model'),
        'checkpoint_goal': payload.get('goal'),
        'checkpoint_training_mode': payload.get('training_mode'),
        'source_commit': config.get('source_commit'),
    }

    del model, flop_input, speed_input, payload, state_dict
    if PROFILE_DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [ ]:
available_experiments = []
for config_path in sorted(BENCHMARK_ROOT.glob('*/config.json')):
    experiment_dir = config_path.parent
    if list((experiment_dir / 'runs').glob('seed_*/best_model.pth')):
        available_experiments.append(experiment_dir)

if not available_experiments:
    raise FileNotFoundError(f'No completed checkpoints found under {BENCHMARK_ROOT}')

selection = EXPERIMENT_FOLDER.strip().casefold()
if selection == 'all':
    experiment_dirs = available_experiments
elif selection == 'latest':
    experiment_dirs = [available_experiments[-1]]
else:
    exact = [path for path in available_experiments if path.name.casefold() == selection]
    matches = exact or [
        path for path in available_experiments if selection in path.name.casefold()
    ]
    if not matches:
        available_names = '\n'.join(f'  {path.name}' for path in available_experiments)
        raise FileNotFoundError(
            f'No completed experiment matches {EXPERIMENT_FOLDER!r}. Available folders:\n{available_names}'
        )
    experiment_dirs = [matches[-1]]
    if len(matches) > 1:
        print(f'{len(matches)} folders matched; using the newest: {matches[-1].name}')

if MAX_EXPERIMENTS is not None:
    experiment_dirs = experiment_dirs[:MAX_EXPERIMENTS]

inventory = []
for experiment_dir in experiment_dirs:
    config = json.loads((experiment_dir / 'config.json').read_text(encoding='utf-8'))
    inventory.append({
        'experiment': experiment_dir.name,
        'model': config['model'],
        'goal': config['goal'],
        'training_mode': config['training_mode'],
        'image_size': config['image_size'],
        'dataset_channels': config['input_channels'],
        'classes': len(config['classes']),
        'saved_seeds': len(list((experiment_dir / 'runs').glob('seed_*/best_model.pth'))),
    })

inventory_table = pd.DataFrame(inventory)
display(inventory_table)
print(f'Experiments to profile: {len(experiment_dirs)}')

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
session_dir = OUTPUT_ROOT / (
    datetime.now().strftime('%Y%m%d_%H%M%S')
    + f'_batch_{INFERENCE_BATCH_SIZE}_complexity'
)
session_dir.mkdir(parents=True, exist_ok=False)

results = []
errors = []
for index, experiment_dir in enumerate(experiment_dirs, start=1):
    print(f'[{index}/{len(experiment_dirs)}] {experiment_dir.name}')
    try:
        results.append(profile_experiment(experiment_dir))
    except Exception as exc:
        errors.append({
            'experiment': experiment_dir.name,
            'error_type': type(exc).__name__,
            'error': str(exc),
            'traceback': traceback.format_exc(),
        })
        print(f'  FAILED: {type(exc).__name__}: {exc}')

results_table = pd.DataFrame(results).sort_values(
    ['goal', 'model', 'training_mode'], ignore_index=True,
) if results else pd.DataFrame()
errors_table = pd.DataFrame(errors)

results_table.to_csv(session_dir / 'model_complexity.csv', index=False)
(session_dir / 'model_complexity.json').write_text(
    json.dumps(results, indent=2) + '\n', encoding='utf-8',
)
if not errors_table.empty:
    errors_table.to_csv(session_dir / 'errors.csv', index=False)
    (session_dir / 'errors.json').write_text(
        json.dumps(errors, indent=2) + '\n', encoding='utf-8',
    )

metadata = {
    'created_at': datetime.now().astimezone().isoformat(),
    'workspace': WORKSPACE_ROOT.name,
    'benchmark_root': 'benchmark_outputs',
    'output_root': (
        'model_complexity_batch_outputs'
        if OUTPUT_ROOT == WORKSPACE_ROOT / 'model_complexity_batch_outputs' else str(OUTPUT_ROOT)
    ),
    'representative_seed_requested': REPRESENTATIVE_SEED,
    'inference_batch_size': INFERENCE_BATCH_SIZE,
    'profile_device': str(PROFILE_DEVICE),
    'flop_backend': FLOP_BACKEND,
    'fps_warmup_iterations': FPS_WARMUP_ITERATIONS,
    'fps_timed_iterations_per_repeat': FPS_TIMED_ITERATIONS,
    'fps_repeats': FPS_REPEATS,
    'torch_version': torch.__version__,
    'fvcore_version': getattr(fvcore, '__version__', None),
    'python_version': platform.python_version(),
    'experiments_requested': len(experiment_dirs),
    'experiments_completed': len(results),
    'experiments_failed': len(errors),
}
(session_dir / 'profile_metadata.json').write_text(
    json.dumps(metadata, indent=2) + '\n', encoding='utf-8',
)

report_columns = [
    'experiment', 'model', 'goal', 'training_mode', 'input_shape',
    'total_parameters', 'trainable_parameters', 'frozen_parameters',
    'parameter_storage_MiB', 'checkpoint_file_MiB', 'GFLOPs_per_image',
    'FPS_mean', 'FPS_std', 'batch_latency_ms_mean', 'batch_latency_ms_std',
    'per_image_latency_ms_mean', 'per_image_latency_ms_std',
    'peak_inference_memory_MiB',
]
if results_table.empty:
    readable_table = 'No models were profiled successfully.'
else:
    readable_table = results_table[report_columns].to_string(
        index=False,
        formatters={
            'parameter_storage_MiB': lambda value: f'{value:.2f}',
            'checkpoint_file_MiB': lambda value: f'{value:.2f}',
            'GFLOPs_per_image': lambda value: f'{value:.3f}',
            'FPS_mean': lambda value: f'{value:.2f}',
            'FPS_std': lambda value: f'{value:.2f}',
            'batch_latency_ms_mean': lambda value: f'{value:.3f}',
            'batch_latency_ms_std': lambda value: f'{value:.3f}',
            'per_image_latency_ms_mean': lambda value: f'{value:.3f}',
            'per_image_latency_ms_std': lambda value: f'{value:.3f}',
            'peak_inference_memory_MiB': (
                lambda value: 'n/a' if pd.isna(value) else f'{value:.2f}'
            ),
        },
    )

notes = [
    'MODEL COMPLEXITY AND BATCH INFERENCE PROFILING',
    '',
    'No dataset, training loop, or evaluation was run.',
    'One saved checkpoint was loaded per experiment to verify the exact architecture.',
    'GFLOPs are for a single synthetic image (batch size 1).',
    f'Inference speed batch size: {INFERENCE_BATCH_SIZE}',
    f'FLOP backend: {FLOP_BACKEND}',
    results[0]['flop_convention'] if results else 'No successful profiles.',
    '',
    'checkpoint_file_MiB is the actual serialized best_model.pth size.',
    'parameter_storage_MiB is the in-memory parameter tensor size.',
    'Total parameters describe the complete inference model.',
    'Trainable parameters reflect the saved training mode.', '',
    (
        f'FPS uses {FPS_WARMUP_ITERATIONS} warm-up passes and '
        f'{FPS_REPEATS} repeats of {FPS_TIMED_ITERATIONS} timed passes.'
    ),
    'FPS excludes dataset loading and preprocessing.', '',
    'Batch latency is the time for one complete forward batch.',
    'Per-image latency is batch latency divided by the batch size.', '',
    readable_table,
]
(session_dir / 'MODEL_COMPLEXITY.txt').write_text('\n'.join(notes) + '\n', encoding='utf-8')
status = 'COMPLETE' if not errors else ('PARTIAL' if results else 'FAILED')
(session_dir / 'STATUS.txt').write_text(
    f'{status}\nCompleted: {len(results)}\nFailed: {len(errors)}\n', encoding='utf-8',
)

print(f'Status: {status}')
print(f'Results: {session_dir}')

In [ ]:
if not results_table.empty:
    paper_columns = [
        'goal', 'model', 'training_mode', 'input_shape',
        'total_parameters_M', 'trainable_parameters_M', 'trainable_percent',
        'checkpoint_file_MiB', 'parameter_storage_MiB', 'GFLOPs_per_image',
        'FPS_mean', 'FPS_std', 'batch_latency_ms_mean', 'batch_latency_ms_std',
        'per_image_latency_ms_mean', 'per_image_latency_ms_std',
        'peak_inference_memory_MiB', 'inference_device', 'flop_backend',
    ]
    display(results_table[paper_columns].round({
        'total_parameters_M': 3,
        'trainable_parameters_M': 3,
        'trainable_percent': 3,
        'checkpoint_file_MiB': 2,
        'parameter_storage_MiB': 2,
        'GFLOPs_per_image': 3,
        'FPS_mean': 2,
        'FPS_std': 2,
        'batch_latency_ms_mean': 3,
        'batch_latency_ms_std': 3,
        'per_image_latency_ms_mean': 3,
        'per_image_latency_ms_std': 3,
        'peak_inference_memory_MiB': 2,
    }))
if not errors_table.empty:
    display(errors_table[['experiment', 'error_type', 'error']])

## Reporting guidance

- Report GFLOPs **per image at batch size 1**, along with the input shape and profiler backend.
- Report both total and trainable parameters. Linear probing and full fine-tuning have similar total parameter counts but very different trainable counts.
- Report FPS as mean ± sample standard deviation across the five timing repeats. FPS covers only the model forward pass; data loading and preprocessing are excluded.
- For batch size 1, FPS describes single-image inference. For larger batches, FPS describes throughput rather than the response time for one image.
- Compare models at the same batch size. Report maximum-batch throughput separately if each model uses a different maximum batch.
- Batch latency and per-image latency are both saved; do not label batch latency as single-image latency.
- State the device, input shape, batch size, precision, warm-up count, and timed iteration count with FPS.
- `checkpoint_file_MiB` is the actual saved artifact size. `parameter_storage_MiB` is the theoretical in-memory tensor storage and should not be described as the checkpoint file size.
- Results are architecture properties, so one representative seed is sufficient. Accuracy metrics should still use all five benchmark seeds.
- Do not compare FLOP values calculated by different profiler backends without recalculating them consistently.